# Unit 2: Wildfire Mapping with CNNs

This notebook supports the Unit 2 ILOs:

- prepare multispectral satellite data for deep learning,
- implement a CNN-based segmentation model,
- generate burn scar maps from satellite imagery.

The notebook includes a full PyTorch implementation of a U-Net style model. If a wildfire dataset is not immediately available, placeholder data are generated so the code remains structurally complete.

## How To Read This Notebook

This notebook is meant to be self-contained for students learning wildfire mapping with deep learning. It is not just a coding exercise. It explains why wildfire mapping is treated as semantic segmentation, why multispectral input data are useful, and why a U-Net style CNN is a strong model for this task.

Students should follow the workflow in sequence: first understand the burn scar problem, then inspect how the data are organized, then study the segmentation model, and finally interpret how predictions become a burn scar map. The final map is not produced by the model alone. It also depends on preprocessing, labels, and threshold choices.

## 1. Problem framing

Burn scar mapping is a semantic segmentation task. Each pixel is assigned to one of two classes: burned or unburned.

Typical inputs include Sentinel-2 bands and vegetation or burn-sensitive indices such as NDVI and NBR. The model below learns spatial patterns as well as spectral contrast.

In [ ]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DATA_DIR = Path('data/unit2_wildfire')
DATA_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

In [ ]:
# Example Hugging Face dataset hook.
# from datasets import load_dataset
# ds = load_dataset('your-org/wildfire-burn-scar-segmentation')

class PlaceholderWildfireDataset(Dataset):
    def __init__(self, n_samples=64, image_size=128):
        self.n_samples = n_samples
        self.image_size = image_size
        self.rng = np.random.default_rng(123)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        h = w = self.image_size
        image = self.rng.uniform(0, 1, size=(6, h, w)).astype(np.float32)
        yy, xx = np.ogrid[:h, :w]
        center_y = self.rng.integers(h // 4, 3 * h // 4)
        center_x = self.rng.integers(w // 4, 3 * w // 4)
        radius = self.rng.integers(h // 8, h // 4)
        mask = (((yy - center_y) ** 2 + (xx - center_x) ** 2) <= radius ** 2).astype(np.float32)
        image[3] *= (1 - 0.6 * mask)
        image[5] += 0.5 * mask
        return torch.from_numpy(image), torch.from_numpy(mask[None, ...])

train_ds = PlaceholderWildfireDataset(n_samples=80)
val_ds = PlaceholderWildfireDataset(n_samples=16)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

In [ ]:
def compute_ndvi(red, nir):
    return (nir - red) / (nir + red + 1e-6)

def compute_nbr(nir, swir):
    return (nir - swir) / (nir + swir + 1e-6)

def normalize_patch(patch, mean=None, std=None):
    if mean is None:
        mean = patch.mean(axis=(1, 2), keepdims=True)
    if std is None:
        std = patch.std(axis=(1, 2), keepdims=True) + 1e-6
    return (patch - mean) / std

# Example channel meaning for a 6-channel tensor:
# [Blue, Green, Red, NIR, NDVI, NBR]

## 2. Full U-Net implementation

The model below is intentionally transparent. Students can inspect encoder, decoder, skip connections, and the final segmentation head.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x):
        return self.conv(self.pool(x))

class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels // 2 + skip_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        diff_y = skip.size(2) - x.size(2)
        diff_x = skip.size(3) - x.size(3)
        x = nn.functional.pad(x, [diff_x // 2, diff_x - diff_x // 2, diff_y // 2, diff_y - diff_y // 2])
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=6, out_channels=1, base_channels=32):
        super().__init__()
        self.inc = DoubleConv(in_channels, base_channels)
        self.down1 = DownBlock(base_channels, base_channels * 2)
        self.down2 = DownBlock(base_channels * 2, base_channels * 4)
        self.down3 = DownBlock(base_channels * 4, base_channels * 8)
        self.bottleneck = DownBlock(base_channels * 8, base_channels * 16)
        self.up1 = UpBlock(base_channels * 16, base_channels * 8, base_channels * 8)
        self.up2 = UpBlock(base_channels * 8, base_channels * 4, base_channels * 4)
        self.up3 = UpBlock(base_channels * 4, base_channels * 2, base_channels * 2)
        self.up4 = UpBlock(base_channels * 2, base_channels, base_channels)
        self.head = nn.Conv2d(base_channels, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.bottleneck(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.head(x)

model = UNet().to(DEVICE)
model

In [ ]:
def dice_score(logits, targets, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return ((2 * intersection + eps) / (union + eps)).mean()

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_dice = 0.0
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        running_dice += dice_score(logits.detach(), masks).item() * images.size(0)
    n = len(loader.dataset)
    return running_loss / n, running_dice / n

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_dice = 0.0
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            logits = model(images)
            loss = criterion(logits, masks)
            running_loss += loss.item() * images.size(0)
            running_dice += dice_score(logits, masks).item() * images.size(0)
    n = len(loader.dataset)
    return running_loss / n, running_dice / n

# for epoch in range(10):
#     train_loss, train_dice = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
#     val_loss, val_dice = validate(model, val_loader, criterion, DEVICE)
#     print(epoch, train_loss, train_dice, val_loss, val_dice)

In [ ]:
def predict_burn_map(model, image_tensor, device, threshold=0.5):
    model.eval()
    with torch.no_grad():
        logits = model(image_tensor.to(device))
        probs = torch.sigmoid(logits)
        burn_map = (probs > threshold).float()
    return probs.cpu(), burn_map.cpu()

# sample_image, _ = val_ds[0]
# probs, burn_map = predict_burn_map(model, sample_image.unsqueeze(0), DEVICE)

## Discussion questions

1. Why do skip connections help in burn scar segmentation?
2. Which input bands or indices are most useful for fire damage detection?
3. How would cloud contamination affect both labels and predictions?
4. What types of post-processing would improve map usability for emergency analysis?

## Unit 2 Takeaway

Wildfire mapping is a semantic segmentation task because the objective is to classify each pixel as burned or unburned rather than assign one class to a whole scene. A U-Net style CNN is effective because it combines local spectral pattern learning with spatial reconstruction through skip connections.

Students should also understand that the final burn scar map depends on more than the architecture. Input bands, derived indices such as NDVI and NBR, the quality of the labels, normalization, and the choice of prediction threshold all influence the result. The model should therefore be interpreted as one component in a broader hazard mapping workflow.